# 5.3 Best-Effort Traffic Anomaly Detection - Data Understanding and Preparation

This notebook prepares a telecom-grade dataset for detecting premium traffic misrouted into best-effort slices.

In [4]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [5]:
from pathlib import Path
cwd = Path.cwd().resolve()
if cwd.name == 'notebooks' and (cwd.parent / 'data').exists():
    project_dir = cwd.parent
elif (cwd / 'data').exists() and (cwd / 'notebooks').exists():
    project_dir = cwd
elif (cwd / 'Jihed' / 'data').exists():
    project_dir = cwd / 'Jihed'
else:
    raise FileNotFoundError(f'Unable to locate project directories from {cwd}')

ROOT = project_dir
DATA_DIR = ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

ROOT: C:\Users\acer\Desktop\Jihed\-Esprit-PIDATA-4DATA-2026-NetworkSlicing\Jihed
DATA_DIR: C:\Users\acer\Desktop\Jihed\-Esprit-PIDATA-4DATA-2026-NetworkSlicing\Jihed\data
PROCESSED_DIR: C:\Users\acer\Desktop\Jihed\-Esprit-PIDATA-4DATA-2026-NetworkSlicing\Jihed\data\processed


In [6]:
prep_script = DATA_DIR / 'prepare_best_effort_anomaly_data.py'
assert prep_script.exists(), f'Missing script: {prep_script}'
print('Preparation script found:', prep_script)

Preparation script found: C:\Users\acer\Desktop\Jihed\-Esprit-PIDATA-4DATA-2026-NetworkSlicing\Jihed\data\prepare_best_effort_anomaly_data.py


In [7]:
import runpy
_ = runpy.run_path(str(prep_script), run_name='__main__')

Prepared files written to: C:\Users\acer\Desktop\Jihed\-Esprit-PIDATA-4DATA-2026-NetworkSlicing\Jihed\data\processed
{
  "best_effort_slice": 2,
  "train_rows": 8958,
  "test_rows": 8960,
  "anomaly_rate_train": 0.17481580709979908,
  "features_hint": [
    "packet_delay_ms",
    "packet_loss_rate",
    "packet_loss_log10",
    "time_sin",
    "time_cos",
    "critical_service_count",
    "is_gbr",
    "is_non_gbr",
    "is_5g",
    "lte_5g_category"
  ]
}


In [ ]:
train = pd.read_csv(PROCESSED_DIR / 'train_anomaly_detection.csv')
test = pd.read_csv(PROCESSED_DIR / 'test_anomaly_detection.csv')
meta = json.loads((PROCESSED_DIR / 'anomaly_detection_metadata.json').read_text(encoding='utf-8'))
print('train shape:', train.shape)
print('test shape:', test.shape)
print(json.dumps(meta, indent=2))

In [ ]:
display(train.head())
display(train[['packet_delay_ms', 'packet_loss_rate', 'premium_intent', 'is_best_effort_slice', 'anomaly_premium_in_best_effort']].describe())

In [ ]:
summary = train.groupby(['premium_intent', 'is_best_effort_slice', 'anomaly_premium_in_best_effort']).size().reset_index(name='count')
summary['ratio'] = summary['count'] / len(train)
display(summary.sort_values('count', ascending=False))

## Next step
Use the Isolation Forest and Autoencoder notebooks to train anomaly detectors on normal best-effort behavior and identify premium-in-best-effort violations.